# C · Message drafts (live session)

Staff pick an allowed offer for a member and get a ready-to-edit message for one channel (W22). Nothing is sent.

Two paths, same evidence: a **deterministic template** (always works, uses only facts in the evidence) and an **LLM
rewrite** under a grounded prompt. LLM output is rejected, and the template used instead, if it mentions churn, risk or
scores, contains a placeholder, uses a number not in the evidence, or runs over the SMS limit.

Run the template cells first. Section 3 makes one LLM call: run it once and read the output for tone and invented claims.

In [ ]:
from sqlmodel import Session

from anchorai.assistant import drafts
from anchorai.assistant.drafts import DraftIn
from anchorai.db.session import get_engine

session = Session(get_engine())

def show(member_id, offer_id, channel, use_llm=False):
    d = drafts.make_draft(session, DraftIn(member_id=member_id, offer_id=offer_id, channel=channel), use_llm=use_llm)
    print(f"--- {member_id} / {offer_id} / {channel} [{d.source}]")
    if d.subject:
        print("Subject:", d.subject)
    print(d.text)
    print("Evidence:", *d.evidence, sep="\n  - ")
    print()

## 1. Template drafts

What to look for: friendly, short, no prices or dates that the catalogue doesn't state, and nothing about risk.

In [ ]:
show("M0042", "free_pt_session", "call")
show("M0042", "checkin_message", "sms")
show("M0155", "discount_1m", "email")

## 2. A blocked offer is refused

Guardrails apply: drafting a message for an offer the rule base blocks raises `OfferBlocked`.

In [ ]:
from anchorai.reasoning.service import all_facts, trace

blocked = next((m, o) for m in all_facts() for o in trace(session, m).blocked if o != "all")
try:
    show(*blocked, "email")
except drafts.OfferBlocked as e:
    print("Refused:", e)

## 3. LLM draft (one call)

Same evidence, rewritten by the model. If the call fails or the output is rejected the cell prints the template (`source: template`).
The endpoint and key are never printed.

In [ ]:
show("M0042", "free_pt_session", "email", use_llm=True)